# Feature Engineering — Real 2025 Flights

This notebook creates and validates the v1 curated model-ready dataset. The production job independently creates v2.

In [ ]:

from pyspark.sql import functions as F

processed_path = (
    "abfss://processed@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/v1/clean_parquet/"
)

real_flights = spark.read.parquet(processed_path)

print("PASS: Real processed dataset loaded")
print("Records:", real_flights.count())

display(real_flights.limit(10))


In [ ]:

model_columns = [
    "FlightDate",
    "Year",
    "Month",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepHour",
    "DayOfWeek",
    "IsWeekend",
    "Quarter",
    "Route",
    "ArrDel15"
]

model_ready = real_flights.select(*model_columns)

# Actual arrival delay must not be a model input.
assert "ArrDelay" not in model_ready.columns

assert model_ready.filter(
    F.col("ArrDel15").isNull()
).count() == 0

assert model_ready.filter(
    ~F.col("ArrDel15").isin(0, 1)
).count() == 0

print("PASS: Real model-ready features created")
print("Records:", model_ready.count())
print("Columns:", len(model_ready.columns))

display(model_ready.limit(10))


In [ ]:

curated_path = (
    "abfss://curated@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/v1/model_ready/"
)

(
    model_ready
    .write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("Month")
    .save(curated_path)
)

print("SUCCESS: Curated model-ready data published!")
print("Path:", curated_path)


In [ ]:

curated_verify = spark.read.parquet(curated_path)

curated_count = curated_verify.count()

assert curated_count == 6879484

assert "ArrDelay" not in curated_verify.columns

assert curated_verify.filter(
    F.col("ArrDel15").isNull()
).count() == 0

print("PASS: Curated dataset verified")
print("Records:", curated_count)
print("PASS: No arrival-delay leakage")
